## Dataset Selection Note

For this analysis, I selected the **KAG Conversion Data** dataset, a public Facebook advertising dataset available on Kaggle. I chose it because it contains real campaign-level marketing performance data with multiple campaign groups and enough observations to support pairwise statistical comparisons.

The dataset includes impressions, clicks, ad spend, total conversions, and approved conversions, which makes it suitable for calculating and comparing marketing metrics such as **CTR, purchase rate, and CPA**. It also contains three main campaign groups with unequal sample sizes, providing a realistic setting for testing statistical significance, effect sizes, multiple-comparison corrections, and power.

Source: Kaggle, **Sales Conversion Optimization / KAG Conversion Data**.

In [2]:
from pathlib import Path

import pandas as pd




In [4]:
# ---------------------------------------------------------
# Paths

BASE_DIR = Path.cwd()

DATA_FILE = (
    BASE_DIR
    / "data"
    / "KAG_conversion_data.csv"
)

print("Working directory:", BASE_DIR)
print("Data file:", DATA_FILE)
print("File exists:", DATA_FILE.exists())




Working directory: /Users/nevena/Ironhack_Labs/week7/lab-statistical-analysis
Data file: /Users/nevena/Ironhack_Labs/week7/lab-statistical-analysis/data/KAG_conversion_data.csv
File exists: True


In [5]:
df = pd.read_csv(DATA_FILE)

In [6]:
df.shape

(1143, 11)

In [8]:
df.columns.tolist()

['ad_id',
 'xyz_campaign_id',
 'fb_campaign_id',
 'age',
 'gender',
 'interest',
 'Impressions',
 'Clicks',
 'Spent',
 'Total_Conversion',
 'Approved_Conversion']

In [9]:
df.head(10)

,ad_id,xyz_campaign_id,fb_campaign_id,age,gender,interest,Impressions,Clicks,Spent,Total_Conversion,Approved_Conversion
0,708746,916,103916,30-34,M,15,7350,1,1.43,2,1
1,708749,916,103917,30-34,M,16,17861,2,1.82,2,0
2,708771,916,103920,30-34,M,20,693,0,0.00,1,0
3,708815,916,103928,30-34,M,28,4259,1,1.25,1,0
4,708818,916,103928,30-34,M,28,4133,1,1.29,1,1
5,708820,916,103929,30-34,M,29,1915,0,0.00,1,1
6,708889,916,103940,30-34,M,15,15615,3,4.77,1,0
7,708895,916,103941,30-34,M,16,10951,1,1.27,1,1
8,708953,916,103951,30-34,M,27,2355,1,1.50,1,0
9,708958,916,103952,30-34,M,28,9502,3,3.16,1,0


In [10]:
df.dtypes

ad_id                    int64
xyz_campaign_id          int64
fb_campaign_id           int64
age                     object
gender                  object
interest                 int64
Impressions              int64
Clicks                   int64
Spent                  float64
Total_Conversion         int64
Approved_Conversion      int64
dtype: object

In [11]:
df.isna().sum()

ad_id                  0
xyz_campaign_id        0
fb_campaign_id         0
age                    0
gender                 0
interest               0
Impressions            0
Clicks                 0
Spent                  0
Total_Conversion       0
Approved_Conversion    0
dtype: int64

In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
# Campaign groups

if "xyz_campaign_id" in df.columns:

    print("\n--- CAMPAIGN GROUPS ---")

    print(
        df["xyz_campaign_id"]
        .value_counts()
        .sort_index()
    )

    print(
        "\nNumber of campaign groups:",
        df["xyz_campaign_id"].nunique(),
    )



--- CAMPAIGN GROUPS ---
xyz_campaign_id
916      54
936     464
1178    625
Name: count, dtype: int64

Number of campaign groups: 3


In [14]:
print("\n--- NUMERIC SUMMARY ---")

print(
    df.describe()
    .T
)


--- NUMERIC SUMMARY ---
                      count           mean            std       min  \
ad_id                1143.0  987261.130359  193992.814738  708746.0   
xyz_campaign_id      1143.0    1067.382327     121.629393     916.0   
fb_campaign_id       1143.0  133783.989501   20500.308622  103916.0   
interest             1143.0      32.766404      26.952131       2.0   
Impressions          1143.0  186732.132983  312762.183208      87.0   
Clicks               1143.0      33.390201      56.892438       0.0   
Spent                1143.0      51.360656      86.908418       0.0   
Total_Conversion     1143.0       2.855643       4.483593       0.0   
Approved_Conversion  1143.0       0.944007       1.737708       0.0   

                           25%         50%          75%           max  
ad_id                777632.50  1121185.00  1121804.500  1.314415e+06  
xyz_campaign_id         936.00     1178.00     1178.000  1.178000e+03  
fb_campaign_id       115716.00   144549.00   144

In [15]:
marketing_columns = [
    "Impressions",
    "Clicks",
    "Spent",
    "Total_Conversion",
    "Approved_Conversion",
]

available_marketing_columns = [
    column
    for column in marketing_columns
    if column in df.columns
]

print("\n--- AVAILABLE MARKETING METRICS ---")
print(available_marketing_columns)


--- AVAILABLE MARKETING METRICS ---
['Impressions', 'Clicks', 'Spent', 'Total_Conversion', 'Approved_Conversion']


In [16]:
print("\n--- ZERO VALUES IN KEY METRICS ---")

for column in available_marketing_columns:

    zero_count = (
        df[column] == 0
    ).sum()

    print(
        f"{column}: {zero_count}"
    )


--- ZERO VALUES IN KEY METRICS ---
Impressions: 0
Clicks: 207
Spent: 207
Total_Conversion: 8
Approved_Conversion: 559


In [17]:
print("=== ROW / ID STRUCTURE ===")

print(
    "Rows:",
    len(df)
)

print(
    "Unique ad_id:",
    df["ad_id"].nunique()
)

print(
    "Unique fb_campaign_id:",
    df["fb_campaign_id"].nunique()
)

print(
    "Unique xyz_campaign_id:",
    df["xyz_campaign_id"].nunique()
)


print("\n=== FB CAMPAIGNS PER MAIN CAMPAIGN ===")

print(
    df.groupby("xyz_campaign_id")["fb_campaign_id"]
    .nunique()
)


print("\n=== AGE GROUPS ===")
print(df["age"].value_counts())


print("\n=== GENDER ===")
print(df["gender"].value_counts())


print("\n=== INTEREST CATEGORIES ===")
print(
    df["interest"].nunique()
)

=== ROW / ID STRUCTURE ===
Rows: 1143
Unique ad_id: 1143
Unique fb_campaign_id: 691
Unique xyz_campaign_id: 3

=== FB CAMPAIGNS PER MAIN CAMPAIGN ===
xyz_campaign_id
916      47
936     367
1178    277
Name: fb_campaign_id, dtype: int64

=== AGE GROUPS ===
age
30-34    426
45-49    259
35-39    248
40-44    210
Name: count, dtype: int64

=== GENDER ===
gender
M    592
F    551
Name: count, dtype: int64

=== INTEREST CATEGORIES ===
40


In [18]:
import numpy as np


# ---------------------------------------------------------
# Prepare analysis dataset
# ---------------------------------------------------------

prepared_df = df.copy()


# ---------------------------------------------------------
# Calculate derived marketing metrics
# ---------------------------------------------------------

prepared_df["CTR"] = prepared_df["Clicks"] / prepared_df["Impressions"]

prepared_df["Approved_Conversion_Rate"] = np.where(prepared_df["Clicks"] > 0, prepared_df["Approved_Conversion"] / prepared_df["Clicks"], np.nan)

prepared_df["CPA"] = np.where(prepared_df["Approved_Conversion"] > 0, prepared_df["Spent"] / prepared_df["Approved_Conversion"], np.nan)


# ---------------------------------------------------------
# Inspect derived metrics
# ---------------------------------------------------------

print("\n--- DERIVED METRICS ---")
print(prepared_df[["CTR", "Approved_Conversion_Rate", "CPA"]].describe().T)

print("\n--- UNDEFINED DERIVED VALUES ---")
print(prepared_df[["CTR", "Approved_Conversion_Rate", "CPA"]].isna().sum())


# ---------------------------------------------------------
# Save prepared dataset
# ---------------------------------------------------------

OUTPUT_FILE = BASE_DIR / "data" / "marketing_data.csv"
prepared_df.to_csv(OUTPUT_FILE, index=False)

print(f"\nPrepared dataset saved to: {OUTPUT_FILE}")


--- DERIVED METRICS ---
                           count       mean        std  min     25%  \
CTR                       1143.0   0.000164   0.000115  0.0  0.0001   
Approved_Conversion_Rate   936.0   0.109030   0.247032  0.0  0.0000   
CPA                        584.0  40.552819  54.273014  0.0  3.0675   

                                50%        75%         max  
CTR                        0.000160   0.000234    0.001059  
Approved_Conversion_Rate   0.012539   0.066667    2.000000  
CPA                       22.847000  54.005000  352.449999  

--- UNDEFINED DERIVED VALUES ---
CTR                           0
Approved_Conversion_Rate    207
CPA                         559
dtype: int64

Prepared dataset saved to: /Users/nevena/Ironhack_Labs/week7/lab-statistical-analysis/data/marketing_data.csv
